## Notebook 04 — ML Workflow: Feature Setup & Train/Validation Split

---

This notebook begins the structured ML pipeline:
1. Load and merge train data, apply >80% missing-column drop from Notebook 03
2. Remove pure identifier columns (`TransactionID`)
3. Stratified 80/20 train-validation split on `isFraud`
4. Print shapes and fraud % for both splits

> **No preprocessing or imputation is performed at this stage.**

---
## 📦 Step 0 — Import Libraries & Load Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../Datasets/'

print('Loading and merging datasets...')
train_transaction = pd.read_csv(DATA_PATH + 'train_transaction.csv')
train_identity    = pd.read_csv(DATA_PATH + 'train_identity.csv')
train_raw = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
del train_transaction, train_identity
print(f'✅ Merged train loaded  — shape: {train_raw.shape}')

# Re-apply >80% missing column drop (consistent with Notebook 03)
missing_pct  = train_raw.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 80].index.tolist()
train = train_raw.drop(columns=cols_to_drop)
del train_raw
print(f'✅ After >80% drop      — shape: {train.shape}  ({len(cols_to_drop)} columns removed)')

Loading and merging datasets...
Merged train loaded  — shape: (590540, 434)
After >80% drop      — shape: (590540, 359)  (74 columns removed)


---
## 🗑️ Step 1 — Remove Pure Identifier Columns

In [2]:
ID_COLS = ['TransactionID']

# Only drop if they exist
id_cols_present = [c for c in ID_COLS if c in train.columns]
train = train.drop(columns=id_cols_present)

print('=' * 50)
print('         IDENTIFIER COLUMN REMOVAL           ')
print('=' * 50)
print(f'  Columns removed : {id_cols_present}')
print(f'  Columns kept    : {train.shape[1]}')
print(f'  TransactionDT   : KEPT (may carry time signal)')
print('=' * 50)

         IDENTIFIER COLUMN REMOVAL           
  Columns removed : ['TransactionID']
  Columns kept    : 359
  TransactionDT   : KEPT (may carry time signal)


---
## ✂️ Step 2 — Stratified 80/20 Train-Validation Split

In [3]:
TARGET = 'isFraud'

X = train.drop(columns=[TARGET])
y = train[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print('✅ Stratified split complete.')

Stratified split complete.


---
## 📊 Step 3 — Print Split Shapes & Fraud Distribution

In [4]:
train_fraud_pct = y_train.mean() * 100
val_fraud_pct   = y_val.mean()   * 100

train_fraud_n   = y_train.sum()
val_fraud_n     = y_val.sum()

print('=' * 62)
print('         TRAIN / VALIDATION SPLIT SUMMARY              ')
print('=' * 62)
print(f'  {"":30s} {"Train":>12} {"Validation":>12}')
print(f'  {"-"*58}')
print(f'  {"Rows":<30} {len(X_train):>12,} {len(X_val):>12,}')
print(f'  {"Features":<30} {X_train.shape[1]:>12} {X_val.shape[1]:>12}')
print(f'  {"Fraud Count (1)":<30} {int(train_fraud_n):>12,} {int(val_fraud_n):>12,}')
print(f'  {"Non-Fraud Count (0)":<30} {int(len(y_train)-train_fraud_n):>12,} {int(len(y_val)-val_fraud_n):>12,}')
print(f'  {"Fraud Percentage":<30} {train_fraud_pct:>11.2f}% {val_fraud_pct:>11.2f}%')
print('=' * 62)

print(f'\n  ✅ Train shape      : {X_train.shape}')
print(f'  ✅ Validation shape : {X_val.shape}')
print(f'  ✅ Train fraud %    : {train_fraud_pct:.2f}%')
print(f'  ✅ Val fraud %      : {val_fraud_pct:.2f}%')
print(f'\n  Stratification check — difference: {abs(train_fraud_pct - val_fraud_pct):.4f}%  ✅')

         TRAIN / VALIDATION SPLIT SUMMARY              
                                        Train   Validation
  ----------------------------------------------------------
  Rows                                472,432      118,108
  Features                                358          358
  Fraud Count (1)                      16,530        4,133
  Non-Fraud Count (0)                 455,902      113,975
  Fraud Percentage                      3.50%        3.50%

  Train shape      : (472432, 358)
  Validation shape : (118108, 358)
  Train fraud %    : 3.50%
  Val fraud %      : 3.50%
  Stratification check — difference: 0.0004%
